# Jupiter to automize clustering for Cats

In [1]:
import pandas as pd
import numpy as np
import sklearn.metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import autosklearn.classification

## I. Open the datasets

In [2]:
df = pd.read_csv('../data/OutCatdata.csv', na_filter= False)
df = df.drop("Unnamed: 0", axis= 1)
dfQuanti = pd.read_csv('../data/OutCatdataQuantitativ.csv', na_filter= False)
dfQuanti = dfQuanti.drop("Unnamed: 0", axis= 1)

/tmp/ipykernel_201037/4248739503.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/OutCatdata.csv', na_filter= False)
/tmp/ipykernel_201037/4248739503.py:3: DtypeWarning: Columns (9,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  dfQuanti = pd.read_csv('../data/OutCatdataQuantitativ.csv', na_filter= False)


/!\\ the NA values are dropped /!\\

## first look at the data

In [3]:
df

,event.id,timestamp,location.long,location.lat,animal.id,animal.life.stage,animal.reproductive.condition,animal.sex,Hunt,N.pray,Hrs.indors,N.neigbours,StartDate,StartHours,EndDate,EndHours
0,6.331585e+08,2015-04-19 01:02:59.000,138.649719,-34.953682,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
1,6.331585e+08,2015-04-19 01:06:59.000,138.649429,-34.953598,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
2,6.331585e+08,2015-04-19 01:09:53.000,138.649429,-34.954014,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
3,6.331585e+08,2015-04-19 01:12:45.000,138.649765,-34.954140,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
4,6.331585e+08,2015-04-19 01:15:37.000,138.649368,-34.954044,Princess,0 years,Sterilized,f,Yes,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057315,2.412263e+09,2015-04-07 03:08:57.000,138.644196,-34.837971,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057316,2.412263e+09,2015-04-07 03:19:06.000,138.644257,-34.837906,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057317,2.412263e+09,2015-04-07 03:29:09.000,138.644531,-34.837643,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057318,2.412263e+09,2015-04-07 03:38:05.000,138.644089,-34.837967,Tiger,5 years,Sterilized,m,Yes,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000


In [4]:
dfQuanti

,event.id,timestamp,location.long,location.lat,animal.id,animal.reproductive.condition,animal.sex,Hunt,N.pray,Hrs.indors,N.neigbours,StartDate,StartHours,EndDate,EndHours,animal.age
0,6.331585e+08,2015-04-19 01:02:59.000,138.649719,-34.953682,Princess,0,1,1,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000,0
1,6.331585e+08,2015-04-19 01:06:59.000,138.649429,-34.953598,Princess,0,1,1,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000,0
2,6.331585e+08,2015-04-19 01:09:53.000,138.649429,-34.954014,Princess,0,1,1,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000,0
3,6.331585e+08,2015-04-19 01:12:45.000,138.649765,-34.954140,Princess,0,1,1,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000,0
4,6.331585e+08,2015-04-19 01:15:37.000,138.649368,-34.954044,Princess,0,1,1,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057315,2.412263e+09,2015-04-07 03:08:57.000,138.644196,-34.837971,Tiger,0,0,1,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000,5
1057316,2.412263e+09,2015-04-07 03:19:06.000,138.644257,-34.837906,Tiger,0,0,1,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000,5
1057317,2.412263e+09,2015-04-07 03:29:09.000,138.644531,-34.837643,Tiger,0,0,1,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000,5
1057318,2.412263e+09,2015-04-07 03:38:05.000,138.644089,-34.837967,Tiger,0,0,1,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000,5


### Split the datasets between classes to predict and data

In [5]:
variables = df.drop(['Hunt'], axis = 1)
variables

,event.id,timestamp,location.long,location.lat,animal.id,animal.life.stage,animal.reproductive.condition,animal.sex,N.pray,Hrs.indors,N.neigbours,StartDate,StartHours,EndDate,EndHours
0,6.331585e+08,2015-04-19 01:02:59.000,138.649719,-34.953682,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
1,6.331585e+08,2015-04-19 01:06:59.000,138.649429,-34.953598,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
2,6.331585e+08,2015-04-19 01:09:53.000,138.649429,-34.954014,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
3,6.331585e+08,2015-04-19 01:12:45.000,138.649765,-34.954140,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
4,6.331585e+08,2015-04-19 01:15:37.000,138.649368,-34.954044,Princess,0 years,Sterilized,f,6,8,1,2015-04-18,16:02:59.000,2015-04-23,19:05:03.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057315,2.412263e+09,2015-04-07 03:08:57.000,138.644196,-34.837971,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057316,2.412263e+09,2015-04-07 03:19:06.000,138.644257,-34.837906,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057317,2.412263e+09,2015-04-07 03:29:09.000,138.644531,-34.837643,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000
1057318,2.412263e+09,2015-04-07 03:38:05.000,138.644089,-34.837967,Tiger,5 years,Sterilized,m,1,10,1,2015-04-01,01:32:24.000,2015-04-07,03:54:13.000


In [6]:
classes = df['Hunt']
classes

0          Yes
1          Yes
2          Yes
3          Yes
4          Yes
          ... 
1057315    Yes
1057316    Yes
1057317    Yes
1057318    Yes
1057319    Yes
Name: Hunt, Length: 1057320, dtype: object

We have 2 resulting df : 

* classes consisting of the true Hunt status
* variables consisting of the factors

## Tentative de coller le TP bêtement

In [7]:
cls = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=120, 
    per_run_time_limit=20, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

*les param minimum pour la tâche* : 

- time_left_for_this_task= 2000 s
- per_run_time_limit=30 cycles
- n_jobs = 16 coeur
- memory_limit = 24 Go

### Génération des jeux de test, validation

pour notre entrainement, nous prenons des proportions de 67% de test et 33% de test

Les méthodes testées sont : 

* Forêt aléatoire
* Latent Dirichlet Allocation
* Multilayered Perceptron
* Baisien naif
* k plus proches voisins 

### Dans un premier temps, nous allons utiliser seulement les données Quantitatives

#### gestion de la suppression des colonnes qualitatives

In [8]:
dfQuali = variables.drop(["event.id","timestamp","location.long","location.lat",
                          "animal.id","StartDate","StartHours","EndDate","EndHours"],
                         axis = 1).astype('category')

Crée le jeu de test qualitatif

In [9]:
variables_trainQ, variables_testQ, classes_trainQ, classes_testQ = train_test_split(dfQuali, classes, test_size = 0.33, random_state=0)

application des paramêtre afin de crée le modèle

In [10]:
cls.fit(variables_trainQ, classes_trainQ, dataset_name = 'OutCatdata')

/home/amouroux/miniforge3/envs/fouille/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35919 instead
  warnings.warn(


[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 412 not found
[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 102 not found
[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 367 not found
[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 262 not found
[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 37 not found
[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 605 not found
[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 88 not found
[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 426 not found
[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 524 not found
[WARNING] [2025-04-18 18:57:37,250:Client-AutoMLSMBO(1)::OutCatdata] Configuration 173 not found
[WARNING] [2025-04-18 18:57:37,2

Process pynisher function call:
Traceback (most recent call last):
  File "/home/amouroux/miniforge3/envs/fouille/lib/python3.9/site-packages/sklearn/utils/_encode.py", line 132, in _unique_python
    uniques = sorted(uniques_set)
TypeError: '<' not supported between instances of 'str' and 'int'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/amouroux/miniforge3/envs/fouille/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/amouroux/miniforge3/envs/fouille/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/amouroux/miniforge3/envs/fouille/lib/python3.9/site-packages/pynisher/limit_function_call.py", line 133, in subprocess_func
    return_value = ((func(*args, **kwargs), 0))
  File "/home/amouroux/miniforge3/envs/fouille/lib/python3.9/site-packages/autosklearn/smbo.py", line 160, in _calculate_metafeatu

[WARNING] [2025-04-18 18:57:38,765:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2025-04-18 18:57:39,092:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2025-04-18 18:57:39,204:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2025-04-18 18:57:39,370:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2025-04-18 18:57:39,509:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2025-04-18 18:57:39,660:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2025-04-18 18:57:41,171:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2025-04-18 18:57:42,237:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2025-04-18 18:57:43,245:Client-EnsembleBuilder] No runs were available to build an ensemble from
[WARNING] [2025-04-18 18:57:

AutoSklearnClassifier(ensemble_class=<class 'autosklearn.ensembles.ensemble_selection.EnsembleSelection'>,
                      include={'classifier': ['random_forest', 'lda', 'mlp',
                                              'gaussian_nb',
                                              'k_nearest_neighbors']},
                      memory_limit=24576, n_jobs=16, per_run_time_limit=20,
                      time_left_for_this_task=120)

Pour l'instant, pn voit que le modèle est claqué :(

	pire que tout, il fonctionne pas 

In [11]:
cls.leaderboard()

,rank,ensemble_weight,type,cost,duration
model_id,,,,,
1,1,1.0,<NA>,<NA>,<NA>


In [12]:
predictions = list(cls.predict(variables_testQ))

précision : 

In [13]:
print("Accuracy score:", sklearn.metrics.accuracy_score(np.array(classes_testQ), predictions))

Accuracy score: 0.14220041499959876


table des stats

In [14]:
print( sklearn.metrics.classification_report(classes_testQ, predictions) )

/home/amouroux/miniforge3/envs/fouille/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/amouroux/miniforge3/envs/fouille/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

          NA       0.14      1.00      0.25     49616
          No       0.00      0.00      0.00     49743
         Yes       0.00      0.00      0.00    249557

    accuracy                           0.14    348916
   macro avg       0.05      0.33      0.08    348916
weighted avg       0.02      0.14      0.04    348916



/home/amouroux/miniforge3/envs/fouille/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


décevant : on a un précision catastrophique.

Nous allons regarder la matrice de confusion pour potentiellement observer quel groupe est le mieu prédit

In [15]:
np.round( confusion_matrix(classes_testQ, predictions), 3)

array([[ 49616,      0,      0],
       [ 49743,      0,      0],
       [249557,      0,      0]])

Le problème est clair : on prédit tout en une classe